# Meshlab Processing

In this step, we fix and simplify the mesh file exported from FIJI. We invert face orientation, reduce the number of faces with Quadric Edge Collapse Decimation, and apply Laplacian smoothing to get rid of step artifacts. When you run this notebook, a window will pop up to select your OBJ file. The export of this notebook is a simplified and smoothed mesh file (your_filename_simplified.obj). 

We found that mesh files from FIJI were typically inverted -- mesh files usually have an orientation, these are known as "normals." If they are inverted, the mesh may appear black in the Meshlab GUI. Quadric Edge Collapse Decimation is a well-used method to simplify a mesh while preserving boundary and normals. Laplacian smooth is a well-used method to smooth mesh surfaces and as a step to fix artifacts from previous steps. All of the mesh processing steps are available in the GUI of Meshlab under the same names. 

In [1]:
# We use the library pymeshlab for mesh processing  
import pymeshlab
import os
from tkinter import filedialog
from pathlib import Path

In [5]:
# Add path to the .obj file 
#obj_file = r"F:\Megumi\Dropbox (DBOX-EQS1)\Library (ECAD)\EH4256-LeftUpper-PCW8.4\ECAD\Airway_Surface\EpcamSegmentation_Handle.ome.tif.obj" 

# Hide the root Tkinter window
#Tk().withdraw()

# Select the .obj file with a pop-up window
obj_file = filedialog.askopenfilename(initialdir = os.sep,title = "Select your OBJ file", filetypes = (("Wavefront OBJ","*.obj"),("all files","*.*")))

# Initialize the pymeshlab MeshSet 
ms = pymeshlab.MeshSet()

# Clear the MeshSet object
ms.clear()

# Load the file 
ms.load_new_mesh(obj_file) 

# Get the number of faces 
num_faces = ms.current_mesh().face_number()

# Calculate the target number of faces to have
target_faces = num_faces // 2

# Consider closing holes with less than 100 
#ms.apply_filter("meshing_close_holes", maxholesize = 100)

# Invert Face Orientation
#ms.apply_filter("meshing_invert_face_orientation")

# Apply Quadric Edge Collapse Decimation 
ms.apply_filter("meshing_decimation_quadric_edge_collapse", targetfacenum=target_faces) 

# Apply Laplacian Smooth
ms.apply_filter("apply_coord_laplacian_smoothing")

# Construct the output file path by renaming with "_simplified.obj" extension
output_file = obj_file.replace('.obj', '_simplifiedby2.obj')
    
# Save the simplified mesh as a new OBJ file
ms.save_current_mesh(output_file)
        
print(f"Mesh '{obj_file}' simplified and saved as '{output_file}'")

Mesh 'F:/Megumi/Dropbox (DBOX-EQS1)/Megumi-Cadisha/Library (ECAD)/Twin102-P12/surface/cleaned_segmentation.tif_simplified_simplified_flipped.obj' simplified and saved as 'F:/Megumi/Dropbox (DBOX-EQS1)/Megumi-Cadisha/Library (ECAD)/Twin102-P12/surface/cleaned_segmentation.tif_simplified_simplified_flipped_simplifiedby2.obj'


The next step is to edit the mesh as necessary (artifact removal, branch identification) in the AutoCad Software Meshmixer. If you import the mesh in Meshmixer, and the mesh appears pink with errors, your face normals are flipped. For the moment, the workaround is to open it in the Meshlab software (the mesh should appear gray if ok and BLACK if normals are flipped, if it is BLACK) go to File > Export mesh as ... .obj (UNCHECK Normal) and save. This saves the mesh surface WITHOUT preserving normals. 

Unfortunately Meshmixer is no longer being updated by Autodesk. You may run into some problems. We have an error report message that pops up when you start the software -- "A software problem has caused Meshmixer to close unexpectedly" that if you IGNORE (DON'T CLOSE THE WINDOW), you can still drag and drop your file and use as necessary. 

You might choose to use another mesh-editing software such as Blender that still regularly receive updates and are maintained but may provide different functionalities and implementations. 

In [ ]:
import os
import pymeshlab
from tkinter import filedialog, Tk

# Hide the root Tkinter window
Tk().withdraw()

# Open a file dialog to select the OBJ file
obj_file = filedialog.askopenfilename(
    initialdir=os.sep,
    title="Select your OBJ file",
    filetypes=(("Wavefront OBJ", "*.obj"), ("all files", "*.*"))
)

if obj_file:
    # Initialize the MeshSet
    ms = pymeshlab.MeshSet()

    # Clear the MeshSet object (optional, as MeshSet is already empty upon initialization)
    ms.clear()

    # Load the mesh file
    ms.load_new_mesh(obj_file)

    # Apply the "Get Geometric Measures" filter
    ms.apply_filter('get_geometric_measures')

    # Get the geometric measures results
    measures = ms.current_mesh().geometric_measures()

    # Define the output file name based on the input OBJ file name
    output_file = os.path.splitext(obj_file)[0] + '_geometric_measures.txt'

    # Write the measures to a text file
    with open(output_file, 'w') as f:
        for key, value in measures.items():
            f.write(f'{key}: {value}\n')

    print(f'Geometric measures have been saved to {output_file}')
else:
    print("No file selected.")